# Lab 2: Logistic Regression -- From Scratch to Practice

In this notebook, you will:
1. Implement logistic regression (binary) from scratch using gradient ascent
2. Apply it to a real binary classification problem (customer churn)
3. Extend to multiclass classification (MNIST digits)
4. Implement and compare L1/L2 regularization
5. Tune hyperparameters via cross-validation

**Prerequisites:** Modules 3 and 4 (Logistic Regression concepts and mathematics)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit as sigmoid
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, classification_report)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Part 1: Binary Logistic Regression from Scratch

From Module 4, the gradient of the log-likelihood is:

$$\nabla_{\boldsymbol{\beta}} \ell = \mathbf{X}^T(\mathbf{y} - \mathbf{p})$$

where $p_i = \sigma(\mathbf{x}_i^T\boldsymbol{\beta})$. We use gradient ascent to maximize.

In [ ]:
class LogisticRegressionFromScratch:
    """Binary logistic regression via gradient ascent on log-likelihood."""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, 
                 penalty=None, C=1.0, tol=1e-6):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.penalty = penalty  # None, 'l1', or 'l2'
        self.C = C  # Inverse regularization strength
        self.tol = tol
        self.beta = None
        self.loss_history = []
    
    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def _log_likelihood(self, X, y, beta):
        z = X @ beta
        ll = np.sum(y * z - np.log(1 + np.exp(np.clip(z, -500, 500))))
        # Add regularization penalty
        if self.penalty == 'l2':
            ll -= (1 / (2 * self.C)) * np.sum(beta[1:] ** 2)
        elif self.penalty == 'l1':
            ll -= (1 / self.C) * np.sum(np.abs(beta[1:]))
        return ll / X.shape[0]
    
    def fit(self, X, y):
        n, p = X.shape
        X_aug = np.column_stack([np.ones(n), X])
        self.beta = np.zeros(p + 1)
        self.loss_history = []
        
        for i in range(self.n_iter):
            # Predictions
            p_hat = self._sigmoid(X_aug @ self.beta)
            
            # Gradient of log-likelihood
            gradient = (1 / n) * X_aug.T @ (y - p_hat)
            
            # Regularization gradient (don't regularize intercept)
            if self.penalty == 'l2':
                reg_grad = np.zeros_like(self.beta)
                reg_grad[1:] = -(1 / self.C) * self.beta[1:]
                gradient += (1 / n) * reg_grad
            elif self.penalty == 'l1':
                reg_grad = np.zeros_like(self.beta)
                reg_grad[1:] = -(1 / self.C) * np.sign(self.beta[1:])
                gradient += (1 / n) * reg_grad
            
            # Gradient ascent update
            self.beta += self.lr * gradient
            
            # Track loss
            ll = self._log_likelihood(X_aug, y, self.beta)
            self.loss_history.append(ll)
            
            # Convergence check
            if np.linalg.norm(gradient) < self.tol:
                break
        
        return self
    
    def predict_proba(self, X):
        n = X.shape[0]
        X_aug = np.column_stack([np.ones(n), X])
        return self._sigmoid(X_aug @ self.beta)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)
    
    @property
    def intercept(self):
        return self.beta[0]
    
    @property
    def coefficients(self):
        return self.beta[1:]

### Test on synthetic binary data

In [ ]:
# Generate 2D binary classification data
from sklearn.datasets import make_classification

X_bin, y_bin = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    random_state=42, n_clusters_per_class=1, class_sep=1.5
)

# Standardize
scaler = StandardScaler()
X_bin_std = scaler.fit_transform(X_bin)

# Fit our model
log_model = LogisticRegressionFromScratch(learning_rate=1.0, n_iterations=500)
log_model.fit(X_bin_std, y_bin)

# Evaluate
y_pred = log_model.predict(X_bin_std)
print(f"Accuracy: {accuracy_score(y_bin, y_pred):.4f}")
print(f"Coefficients: beta0={log_model.intercept:.3f}, beta1={log_model.coefficients[0]:.3f}, beta2={log_model.coefficients[1]:.3f}")

In [ ]:
# Visualize decision boundary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision boundary plot
ax = axes[0]
xx, yy = np.meshgrid(np.linspace(X_bin_std[:, 0].min()-1, X_bin_std[:, 0].max()+1, 200),
                      np.linspace(X_bin_std[:, 1].min()-1, X_bin_std[:, 1].max()+1, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = log_model.predict_proba(grid).reshape(xx.shape)

ax.contourf(xx, yy, probs, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.3)
ax.contour(xx, yy, probs, levels=[0.5], colors='k', linewidths=2)
ax.scatter(X_bin_std[y_bin==0, 0], X_bin_std[y_bin==0, 1], c='blue', alpha=0.6, label='Class 0')
ax.scatter(X_bin_std[y_bin==1, 0], X_bin_std[y_bin==1, 1], c='red', alpha=0.6, label='Class 1')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.set_title('Decision Boundary')
ax.legend()

# Convergence
axes[1].plot(log_model.loss_history)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Log-Likelihood')
axes[1].set_title('Training Convergence')

plt.tight_layout()
plt.show()

### Verify against scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression

sk_log = LogisticRegression(penalty=None, max_iter=1000)
sk_log.fit(X_bin_std, y_bin)

print("Comparison (ours vs sklearn):")
print(f"  Intercept:  {log_model.intercept:.4f} vs {sk_log.intercept_[0]:.4f}")
print(f"  Coef 1:     {log_model.coefficients[0]:.4f} vs {sk_log.coef_[0, 0]:.4f}")
print(f"  Coef 2:     {log_model.coefficients[1]:.4f} vs {sk_log.coef_[0, 1]:.4f}")

## Part 2: Binary Classification -- Customer Churn

Predict which telecom customers will churn (cancel service).

In [ ]:
# Generate realistic churn data
np.random.seed(42)
n = 1000

# Features
tenure = np.random.exponential(24, n)  # months
monthly_charges = np.random.uniform(20, 110, n)
contract_monthly = np.random.binomial(1, 0.5, n)  # 1=month-to-month
tech_support = np.random.binomial(1, 0.3, n)  # 1=has tech support
senior = np.random.binomial(1, 0.15, n)
dependents = np.random.binomial(1, 0.3, n)
online_security = np.random.binomial(1, 0.4, n)
paperless = np.random.binomial(1, 0.6, n)

# Churn probability (logistic model)
logit = (-2.0 
         - 0.03 * tenure 
         + 0.02 * monthly_charges 
         + 1.5 * contract_monthly 
         - 0.8 * tech_support
         + 0.3 * senior
         - 0.4 * dependents
         - 0.6 * online_security
         + 0.2 * paperless)
prob_churn = 1 / (1 + np.exp(-logit))
churn = np.random.binomial(1, prob_churn)

churn_df = pd.DataFrame({
    'tenure': tenure, 'monthly_charges': monthly_charges,
    'contract_monthly': contract_monthly, 'tech_support': tech_support,
    'senior': senior, 'dependents': dependents,
    'online_security': online_security, 'paperless': paperless,
    'churn': churn
})

print(f"Churn rate: {churn.mean():.1%}")
print(f"\nFeature statistics:")
print(churn_df.describe().round(2))

In [ ]:
# Prepare data
feature_cols = ['tenure', 'monthly_charges', 'contract_monthly', 
                'tech_support', 'senior', 'dependents', 
                'online_security', 'paperless']

X_churn = churn_df[feature_cols].values
y_churn = churn_df['churn'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn
)

# Standardize
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# Fit our model
churn_model = LogisticRegressionFromScratch(learning_rate=1.0, n_iterations=1000)
churn_model.fit(X_train_std, y_train)

# Evaluate
y_pred = churn_model.predict(X_test_std)
y_proba = churn_model.predict_proba(X_test_std)

print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

In [ ]:
# Coefficient interpretation
print("Feature importance (odds ratios):")
print(f"{'Feature':<20} {'Coefficient':>12} {'Odds Ratio':>12} {'Interpretation'}")
print("-" * 80)
for feat, coef in sorted(zip(feature_cols, churn_model.coefficients), key=lambda x: -abs(x[1])):
    or_val = np.exp(coef)
    direction = 'increases' if coef > 0 else 'decreases'
    print(f"{feat:<20} {coef:>12.4f} {or_val:>12.4f} {direction} churn risk")

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].fill_between(fpr, tpr, alpha=0.1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Retained', 'Churned'], ax=axes[1], cmap='Blues'
)
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

## Part 3: Regularization Comparison

Compare L1 (feature selection) vs L2 (coefficient shrinkage) on the churn data.

In [ ]:
# L1 vs L2 coefficient paths
C_values = np.logspace(-2, 2, 20)
l1_coefs = []
l2_coefs = []

for C in C_values:
    # L1
    m_l1 = LogisticRegression(penalty='l1', C=C, solver='saga', max_iter=5000)
    m_l1.fit(X_train_std, y_train)
    l1_coefs.append(m_l1.coef_[0])
    
    # L2
    m_l2 = LogisticRegression(penalty='l2', C=C, max_iter=5000)
    m_l2.fit(X_train_std, y_train)
    l2_coefs.append(m_l2.coef_[0])

l1_coefs = np.array(l1_coefs)
l2_coefs = np.array(l2_coefs)

# Plot coefficient paths
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, feat in enumerate(feature_cols):
    axes[0].plot(np.log10(C_values), l1_coefs[:, i], label=feat)
    axes[1].plot(np.log10(C_values), l2_coefs[:, i], label=feat)

axes[0].set_xlabel('log10(C) [less regularization -->]')
axes[0].set_ylabel('Coefficient')
axes[0].set_title('L1 Regularization Path (Lasso)')
axes[0].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[0].legend(fontsize=8, loc='upper left')

axes[1].set_xlabel('log10(C) [less regularization -->]')
axes[1].set_ylabel('Coefficient')
axes[1].set_title('L2 Regularization Path (Ridge)')
axes[1].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[1].legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

print("\nL1 at strong regularization (C=0.01) -- features selected:")
m_sparse = LogisticRegression(penalty='l1', C=0.01, solver='saga', max_iter=5000)
m_sparse.fit(X_train_std, y_train)
for feat, coef in zip(feature_cols, m_sparse.coef_[0]):
    if abs(coef) > 1e-6:
        print(f"  {feat}: {coef:.4f}")

## Part 4: Hyperparameter Tuning via Cross-Validation

In [ ]:
# Grid search
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
}

grid_search = GridSearchCV(
    LogisticRegression(solver='saga', max_iter=5000),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1
)
grid_search.fit(X_train_std, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")
print(f"Test AUC: {roc_auc:.4f}")

# Results table
results_df = pd.DataFrame(grid_search.cv_results_)
pivot = results_df.pivot_table(
    values='mean_test_score', 
    index='param_C', 
    columns='param_penalty'
)
print("\nCV AUC by (C, penalty):")
print(pivot.round(4))

## Part 5: Multiclass Classification (MNIST Digits)

Extend logistic regression to recognize handwritten digits 0-9.

In [ ]:
from sklearn.datasets import load_digits

# Load 8x8 digit images
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Dataset: {X_digits.shape[0]} samples, {X_digits.shape[1]} features (8x8 pixels)")
print(f"Classes: {np.unique(y_digits)}")

# Show sample digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')
    ax.set_title(f'Label: {y_digits[i]}')
    ax.axis('off')
plt.suptitle('Sample Digits')
plt.tight_layout()
plt.show()

In [ ]:
# Train multiclass logistic regression
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42, stratify=y_digits
)

# Standardize
scaler_d = StandardScaler()
X_train_d_std = scaler_d.fit_transform(X_train_d)
X_test_d_std = scaler_d.transform(X_test_d)

# OvR (One-vs-Rest)
ovr_model = LogisticRegression(multi_class='ovr', penalty='l2', C=1.0, max_iter=5000)
ovr_model.fit(X_train_d_std, y_train_d)
ovr_acc = ovr_model.score(X_test_d_std, y_test_d)

# Multinomial (Softmax)
softmax_model = LogisticRegression(multi_class='multinomial', penalty='l2', C=1.0, max_iter=5000)
softmax_model.fit(X_train_d_std, y_train_d)
softmax_acc = softmax_model.score(X_test_d_std, y_test_d)

print(f"One-vs-Rest accuracy:  {ovr_acc:.4f}")
print(f"Softmax accuracy:      {softmax_acc:.4f}")

In [ ]:
# Confusion matrix for softmax model
y_pred_d = softmax_model.predict(X_test_d_std)

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test_d, y_pred_d, ax=ax, cmap='Blues', colorbar=False
)
ax.set_title('Multiclass Confusion Matrix (Softmax)')
plt.tight_layout()
plt.show()

# Which digits are most confused?
cm = confusion_matrix(y_test_d, y_pred_d)
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(cm.argmax(), cm.shape)
print(f"\nMost confused pair: digit {i} misclassified as {j} ({cm[i,j]} times)")

In [ ]:
# Visualize learned weights for each digit class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axes.flat):
    weights = softmax_model.coef_[k].reshape(8, 8)
    ax.imshow(weights, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'Digit {k} weights')
    ax.axis('off')
plt.suptitle('Learned Feature Weights per Class (what the model looks for)')
plt.tight_layout()
plt.show()

## Part 6: Exercises

1. **Threshold optimization:** For the churn model, find the threshold that maximizes the F1-score. How does it differ from 0.5? Plot precision and recall as a function of threshold.

2. **Newton's method:** Implement IRLS (Module 4) for the binary case. Compare convergence speed (number of iterations) with gradient ascent.

3. **Cost-sensitive classification:** In the churn problem, assume that a false negative (missed churn) costs the company $500 in lost revenue, while a false positive (unnecessary retention offer) costs $50. Find the optimal threshold that minimizes total expected cost.

4. **Feature engineering for MNIST:** The 64 raw pixels give ~96% accuracy. Can you get higher by adding polynomial features (pixel interactions)? What about PCA preprocessing? How does regularization strength need to change with more features?

In [ ]:
# Exercise workspace

# Example: Threshold optimization for churn
thresholds = np.linspace(0.1, 0.9, 50)
f1_scores = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    f1_scores.append(f1)

best_idx = np.argmax(f1_scores)
print(f"Best threshold: {thresholds[best_idx]:.3f} (F1 = {f1_scores[best_idx]:.4f})")

plt.plot(thresholds, f1_scores)
plt.axvline(thresholds[best_idx], color='r', linestyle='--', label=f'Best: {thresholds[best_idx]:.3f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Classification Threshold')
plt.legend()
plt.show()